In [6]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 150)

In [7]:
DATA_PATH = 'pm_datasheet.xlsx'  # change this to wherever you keep the file locally

xl = pd.ExcelFile(DATA_PATH)
pm_summary_raw = pd.read_excel(xl, sheet_name='pm_summary')
eqplist_raw    = pd.read_excel(xl, sheet_name='eqplist')
pm_schedule_raw = pd.read_excel(xl, sheet_name='pm_schedule')

print('pm_summary :', pm_summary_raw.shape)
print('eqplist    :', eqplist_raw.shape)
print('pm_schedule:', pm_schedule_raw.shape)

pm_summary : (503818, 6)
eqplist    : (105856, 33)
pm_schedule: (6, 3)


In [8]:
# --- pm_schedule: interval lookup ---
pm_schedule = pm_schedule_raw.rename(columns={
    'id': 'schedule_id',
    'PM_Schedule_Name': 'schedule_name',
    'Interval_In_Days': 'interval_days'
})
pm_schedule

,schedule_id,schedule_name,interval_days
0,1,MONTHLY,30
1,2,WEEKLY,7
2,3,YEARLY,365
3,4,QUARTERLY,90
4,5,BI-WEEKLY,14
5,6,HALF YEARLY,180


In [9]:
# --- eqplist: equipment master, trimmed to what we need ---
eq_cols = ['id', 'line', 'Section', 'Station', 'System', 'SubSystem', 'Eqp_Name',
           'eqp_no', 'EqpID', 'PM_Required', 'critical', 'make', 'station_zone']

eqplist = eqplist_raw[eq_cols].rename(columns={
    'id': 'eqp_id',
    'Section': 'section',
    'Station': 'station',
    'System': 'system',
    'SubSystem': 'subsystem'
})
eqplist.head()

,eqp_id,line,section,station,system,subsystem,Eqp_Name,eqp_no,EqpID,PM_Required,critical,make,station_zone
0,1,L1,L1A,DSG,AFC,GATE,GATE1,1,DSG-GATE-1,1,1,SAMSUNG,Concourse
1,2,L1,L1A,ATHA,AFC,GATE,GATE11,11,ATHA-GATE-11,1,1,INDRA,Concourse
2,3,L1,L1A,ATHA,AFC,GATE,GATE1,1,ATHA-GATE-1,1,1,INDRA,Concourse
3,6,L1,L1A,ATHA,AFC,GATE,GATE2,2,ATHA-GATE-2,1,1,INDRA,Concourse
4,7,L1,L1A,ATHA,AFC,GATE,GATE22,22,ATHA-GATE-22,1,1,INDRA,Concourse


In [10]:
# --- pm_summary: the core PM completion log ---
pm_summary = pm_summary_raw.rename(columns={
    'eqpRef': 'eqp_id',
    'Schedule': 'schedule_id',
    'Done': 'done_date',
    'done by': 'done_by'
})
pm_summary['done_date'] = pd.to_datetime(pm_summary['done_date'], errors='coerce')

print('Rows with unparseable done_date:', pm_summary['done_date'].isna().sum())
pm_summary.head()

Rows with unparseable done_date: 0


,id,eqp_id,PMRef,schedule_id,done_date,done_by
0,514391,6281,100156,1,2026-06-17,NaN
1,514392,6284,100156,1,2026-06-17,NaN
2,514393,6285,100156,1,2026-06-17,NaN
3,514394,6286,100156,1,2026-06-17,NaN
4,514395,6312,100156,1,2026-06-17,NaN


In [11]:
merged = (pm_summary
    .merge(pm_schedule, on='schedule_id', how='left')
    .merge(eqplist, on='eqp_id', how='left'))

print('Unmatched schedule_id:', merged['schedule_name'].isna().sum())
print('Unmatched eqp_id     :', merged['station'].isna().sum(), '/', len(merged))

# Drop the small number of records whose equipment can't be located —
# they can't be placed into any station/system filter anyway
before = len(merged)
merged = merged.dropna(subset=['station']).copy()
print(f'Dropped {before - len(merged)} unmatched rows ({(before-len(merged))/before:.2%})')
print('Final merged shape:', merged.shape)

Unmatched schedule_id: 0
Unmatched eqp_id     : 801 / 503818
Dropped 801 unmatched rows (0.16%)
Final merged shape: (503017, 20)


## 4. Compliance calculation

For each equipment + schedule-type "track" (e.g. CCTV-1234's monthly PM history),
sorted by date, the **due date for record N is record N-1's completion date + the
scheduled interval**. We compare the actual completion date against that.

- `baseline` — first-ever record for that equipment+schedule combo (nothing to compare against yet)
- `on_time` — completed within a 3-day grace window of the expected due date
- `late` — completed after the grace window

3 days of grace is a starting assumption — tune it once you've shown this to your
superior and confirmed what counts as "on time" in practice.

In [12]:
GRACE_DAYS = 3

merged = merged.sort_values(['eqp_id', 'schedule_id', 'done_date'])
merged['prev_done_date'] = merged.groupby(['eqp_id', 'schedule_id'])['done_date'].shift(1)
merged['expected_due_date'] = merged['prev_done_date'] + pd.to_timedelta(merged['interval_days'], unit='D')
merged['days_late'] = (merged['done_date'] - merged['expected_due_date']).dt.days

conditions = [merged['prev_done_date'].isna(), merged['days_late'] <= GRACE_DAYS]
choices = ['baseline', 'on_time']
merged['compliance_status'] = np.select(conditions, choices, default='late')

merged['compliance_status'].value_counts()

compliance_status
on_time     221022
late        147167
baseline    134828
Name: count, dtype: int64

In [13]:
# Sanity check: look at one equipment's history to confirm the logic makes sense
sample_eqp = merged['eqp_id'].value_counts().index[10]
cols_to_show = ['eqp_id', 'schedule_name', 'done_date', 'prev_done_date',
                'expected_due_date', 'days_late', 'compliance_status']
merged[merged['eqp_id'] == sample_eqp][cols_to_show].head(15)

,eqp_id,schedule_name,done_date,prev_done_date,expected_due_date,days_late,compliance_status
499584,8520,MONTHLY,2025-01-06,NaT,NaT,NaN,baseline
473850,8520,MONTHLY,2025-03-04,2025-01-06,2025-02-05,27.0,late
465755,8520,MONTHLY,2025-03-17,2025-03-04,2025-04-03,-17.0,on_time
455033,8520,MONTHLY,2025-04-01,2025-03-17,2025-04-16,-15.0,on_time
443075,8520,MONTHLY,2025-04-19,2025-04-01,2025-05-01,-12.0,on_time
434960,8520,MONTHLY,2025-05-02,2025-04-19,2025-05-19,-17.0,on_time
410362,8520,MONTHLY,2025-06-09,2025-05-02,2025-06-01,8.0,late
393641,8520,MONTHLY,2025-07-05,2025-06-09,2025-07-09,-4.0,on_time
384982,8520,MONTHLY,2025-07-20,2025-07-05,2025-08-04,-15.0,on_time
375102,8520,MONTHLY,2025-08-02,2025-07-20,2025-08-19,-17.0,on_time


## 5. Build the aggregation table

This is what the Streamlit dashboard will actually query — pre-aggregated by
every filter dimension (station, system, sub-system, schedule type), so the app
doesn't have to crunch 500k rows on every filter click.

In [14]:
# exclude baseline records — there's nothing to score compliance on for those
trackable = merged[merged['compliance_status'] != 'baseline'].copy()

agg = (trackable.groupby(['station', 'system', 'subsystem', 'schedule_name'])
       .agg(total_pm=('compliance_status', 'count'),
            on_time=('compliance_status', lambda x: (x == 'on_time').sum()),
            late=('compliance_status', lambda x: (x == 'late').sum()),
            avg_days_late=('days_late', 'mean'))
       .reset_index())

agg['compliance_pct'] = (agg['on_time'] / agg['total_pm'] * 100).round(1)
agg['avg_days_late'] = agg['avg_days_late'].round(1)

print('Aggregation table shape:', agg.shape)
agg.sort_values('compliance_pct').head(10)

Aggregation table shape: (7028, 9)


,station,system,subsystem,schedule_name,total_pm,on_time,late,avg_days_late,compliance_pct
2960,KKDC,AFC,EC,QUARTERLY,3,0,3,26.0,0.0
2953,KKDA7,TELECOM,SMPS,HALF YEARLY,4,0,4,39.0,0.0
78,AIIM,TELECOM,EPABX,MONTHLY,12,0,12,37.7,0.0
1889,HNOK,TELECOM,CLOCK,HALF YEARLY,2,0,2,121.0,0.0
4001,NCBC,TELECOM,UPS,HALF YEARLY,1,0,1,98.0,0.0
2979,KKDC,TELECOM,RADIO,MONTHLY,4,0,4,73.5,0.0
4031,NDI,TELECOM,PAS/PIDS,MONTHLY,2,0,2,81.0,0.0
2977,KKDC,TELECOM,PAS/PIDS,MONTHLY,94,0,94,206.2,0.0
1322,DSTN,TELECOM,TELE SWITCH,QUARTERLY,2,0,2,93.0,0.0
2974,KKDC,TELECOM,EPABX,MONTHLY,4,0,4,24.0,0.0


In [15]:
# Worst-performing equipment types network-wide (lowest compliance, min 20 records to avoid noise)
worst = (agg[agg['total_pm'] >= 20]
         .groupby('subsystem')
         .apply(lambda d: pd.Series({
             'total_pm': d['total_pm'].sum(),
             'compliance_pct': (d['on_time'].sum() / d['total_pm'].sum() * 100).round(1)
         }))
         .sort_values('compliance_pct')
         .reset_index())
worst.head(15)

/var/folders/97/yd_ylpmd5gl77djbdrmr_y3r0000gn/T/ipykernel_10017/2715637969.py:4: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda d: pd.Series({


,subsystem,total_pm,compliance_pct
0,TIM,60.0,20.0
1,EMERGENCY SWITCH,95.0,42.1
2,SDH/MPLS,96.0,49.0
3,AVM,6323.0,50.8
4,RCTM,3071.0,55.6
5,GATE,43693.0,56.3
6,EPABX,12150.0,57.3
7,SWING BARRIER,721.0,57.4
8,UPS,2215.0,57.5
9,PAS/PIDS,148832.0,58.7


## 6. Export cleaned data

Two files for the Streamlit app:
- `pm_records_clean.csv` — full record-level data (for the overdue list / drill-down view)
- `pm_compliance_agg.csv` — pre-aggregated table (for the fast filter dashboard / heatmap)

In [16]:
output_cols = ['eqp_id', 'EqpID', 'Eqp_Name', 'line', 'section', 'station', 'system',
               'subsystem', 'station_zone', 'critical', 'schedule_name', 'interval_days',
               'done_date', 'done_by', 'expected_due_date', 'days_late', 'compliance_status']

merged[output_cols].to_csv('pm_records_clean.csv', index=False)
agg.to_csv('pm_compliance_agg.csv', index=False)

print('Exported pm_records_clean.csv:', merged.shape[0], 'rows')
print('Exported pm_compliance_agg.csv:', agg.shape[0], 'rows')

Exported pm_records_clean.csv: 503017 rows
Exported pm_compliance_agg.csv: 7028 rows


## Next steps

- Open `pm_compliance_agg.csv` and `pm_records_clean.csv` in the Streamlit app for Module 1's
  filters and heatmap.
- Tune `GRACE_DAYS` once you confirm what "on time" means operationally.
- The 1,537 equipment IDs from `pm_summary` that had no match in `eqplist` — worth asking your
  superior whether `eqplist` is missing some assets, or these are decommissioned/test entries.
- `done_by` is still messy free text — leave it alone for Module 1, it isn't used here. We'll
  tackle name normalization separately when we get to Module 3.

In [21]:
merged[merged['station'] == 'NCBC']

,id,eqp_id,PMRef,schedule_id,done_date,done_by,schedule_name,interval_days,line,section,station,system,subsystem,Eqp_Name,eqp_no,EqpID,PM_Required,critical,make,station_zone,prev_done_date,expected_due_date,days_late,compliance_status
499516,131525,3044,3770,1,2025-01-07,NaN,MONTHLY,30,L6,L6B,NCBC,AFC,GATE,GATE1,1.0,NCBC-GATE-1,1.0,1.0,NaN,Concourse,NaT,NaT,NaN,baseline
484323,131524,3044,8822,1,2025-02-17,NaN,MONTHLY,30,L6,L6B,NCBC,AFC,GATE,GATE1,1.0,NCBC-GATE-1,1.0,1.0,NaN,Concourse,2025-01-07,2025-02-06,11.0,late
468316,131523,3044,12076,1,2025-03-12,NaN,MONTHLY,30,L6,L6B,NCBC,AFC,GATE,GATE1,1.0,NCBC-GATE-1,1.0,1.0,NaN,Concourse,2025-02-17,2025-03-19,-7.0,on_time
454703,131522,3044,14988,1,2025-04-02,NaN,MONTHLY,30,L6,L6B,NCBC,AFC,GATE,GATE1,1.0,NCBC-GATE-1,1.0,1.0,NaN,Concourse,2025-03-12,2025-04-11,-9.0,on_time
429256,131521,3044,20268,1,2025-05-13,NaN,MONTHLY,30,L6,L6B,NCBC,AFC,GATE,GATE1,1.0,NCBC-GATE-1,1.0,1.0,NaN,Concourse,2025-04-02,2025-05-02,11.0,late
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
225049,285655,72565,61069,4,2025-12-30,17041,QUARTERLY,90,L6,L6B,NCBC,TELECOM,RADIO,RADIO-TOWER1,1.0,NCBC-RADIO-TOWER-1,1.0,1.0,NaN,Platform1,2025-09-29,2025-12-28,2.0,on_time
158633,353977,72565,73575,4,2026-03-03,21408,QUARTERLY,90,L6,L6B,NCBC,TELECOM,RADIO,RADIO-TOWER1,1.0,NCBC-RADIO-TOWER-1,1.0,1.0,NaN,Platform1,2025-12-30,2026-03-30,-27.0,on_time
145841,367070,109281,76054,3,2026-03-17,17041,YEARLY,365,L6,L6B,NCBC,IT,IT ROUTER,IT ROUTER1,1.0,NCBC-IT-RTR-1,1.0,1.0,HP,TER,NaT,NaT,NaN,baseline
71976,442201,115637,88104,1,2026-05-03,17041,MONTHLY,30,L6,L6B,NCBC,TELECOM,TER ROOM,NaN,1.0,NCBC-TER ROOM-1,1.0,1.0,NaN,TER,NaT,NaT,NaN,baseline


In [22]:
station_check = merged[merged['station'] == 'NCBC']
station_check[['EqpID', 'subsystem', 'schedule_name', 'done_date', 'prev_done_date',
               'expected_due_date', 'days_late', 'compliance_status']].sort_values('done_date')

,EqpID,subsystem,schedule_name,done_date,prev_done_date,expected_due_date,days_late,compliance_status
499516,NCBC-GATE-1,GATE,MONTHLY,2025-01-07,NaT,NaT,NaN,baseline
499509,NCBC-GATE-12,GATE,MONTHLY,2025-01-07,NaT,NaT,NaN,baseline
499522,NCBC-SMPS-1,SMPS,MONTHLY,2025-01-07,NaT,NaT,NaN,baseline
499520,NCBC-SMPS-2,SMPS,MONTHLY,2025-01-07,NaT,NaT,NaN,baseline
499504,NCBC-RADIO-RAU/ZETRON-3623301,RADIO,MONTHLY,2025-01-07,NaT,NaT,NaN,baseline
...,...,...,...,...,...,...,...,...
2093,NCBC-TELEPHONE-EPABX-1,EPABX,HALF YEARLY,2026-06-16,2025-12-10,2026-06-08,8.0,late
2055,NCBC-GATE-12,GATE,MONTHLY,2026-06-16,2026-05-06,2026-06-05,11.0,late
2044,NCBC-TELEPHONE-DIGITAL-665532,EPABX,MONTHLY,2026-06-16,2026-05-06,2026-06-05,11.0,late
2045,NCBC-SB-1,SWING BARRIER,MONTHLY,2026-06-16,2026-05-06,2026-06-05,11.0,late


In [23]:
merged[merged['days_late'] > 300][['EqpID', 'station', 'schedule_name', 'done_date',
                                     'prev_done_date', 'days_late']].head(20)

,EqpID,station,schedule_name,done_date,prev_done_date,days_late
417327,DSG-TVM-1,DSG,MONTHLY,2025-05-28,2024-04-25,368.0
417331,JLML-TVM-1,JLML,MONTHLY,2025-05-28,2024-04-09,384.0
380460,KP-TVM-1,KP,BI-WEEKLY,2025-07-26,2024-05-24,414.0
380459,KP-TVM-2,KP,BI-WEEKLY,2025-07-26,2024-05-24,414.0
231849,RI-TVM-1,RI,BI-WEEKLY,2025-12-23,2024-05-04,584.0
417345,SHD-TVM-1,SHD,MONTHLY,2025-05-28,2024-03-20,404.0
417344,SHD-TVM-2,SHD,MONTHLY,2025-05-28,2024-04-02,391.0
417342,SHD-TVM-4,SHD,MONTHLY,2025-05-28,2024-03-20,404.0
469991,SHD-GATE-1,SHD,MONTHLY,2025-03-10,2024-04-02,312.0
43652,PALM-AFC-SW-1,PALM,QUARTERLY,2026-05-20,2025-02-27,357.0


In [24]:
merged[merged['system'] == 'AFC']

,id,eqp_id,PMRef,schedule_id,done_date,done_by,schedule_name,interval_days,line,section,station,system,subsystem,Eqp_Name,eqp_no,EqpID,PM_Required,critical,make,station_zone,prev_done_date,expected_due_date,days_late,compliance_status
487238,146387,1,8124,1,2025-02-11,NaN,MONTHLY,30,L1,L1A,DSG,AFC,GATE,GATE1,1.0,DSG-GATE-1,1.0,1.0,SAMSUNG,Concourse,NaT,NaT,NaN,baseline
486869,146386,1,8211,1,2025-02-12,NaN,MONTHLY,30,L1,L1A,DSG,AFC,GATE,GATE1,1.0,DSG-GATE-1,1.0,1.0,SAMSUNG,Concourse,2025-02-11,2025-03-13,-29.0,on_time
473849,146385,1,10862,1,2025-03-04,NaN,MONTHLY,30,L1,L1A,DSG,AFC,GATE,GATE1,1.0,DSG-GATE-1,1.0,1.0,SAMSUNG,Concourse,2025-02-12,2025-03-14,-10.0,on_time
473334,146384,1,10980,1,2025-03-05,NaN,MONTHLY,30,L1,L1A,DSG,AFC,GATE,GATE1,1.0,DSG-GATE-1,1.0,1.0,SAMSUNG,Concourse,2025-03-04,2025-04-03,-29.0,on_time
454588,146383,1,15045,1,2025-04-02,NaN,MONTHLY,30,L1,L1A,DSG,AFC,GATE,GATE1,1.0,DSG-GATE-1,1.0,1.0,SAMSUNG,Concourse,2025-03-05,2025-04-04,-2.0,on_time
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23965,490410,115872,95964,4,2026-05-31,Sanjeeb Kumar Phukan,QUARTERLY,90,L2,L2A,SLTP,AFC,AFC SWITCH,AFC SWITCH2,2.0,SLTP-AFC-SW-2,1.0,1.0,NaN,Concourse,NaT,NaT,NaN,baseline
10972,503428,115882,98375,1,2026-06-11,Vivek Kumar,MONTHLY,30,L1,L1A,NBAA,AFC,GATE,GATE15,15.0,NBAA-GATE-15,1.0,1.0,INDRA,Concourse,NaT,NaT,NaN,baseline
10973,503429,115883,98375,1,2026-06-11,Vivek Kumar,MONTHLY,30,L1,L1A,NBAA,AFC,GATE,GATE16,16.0,NBAA-GATE-16,1.0,1.0,INDRA,Concourse,NaT,NaT,NaN,baseline
384,514020,115888,100124,1,2026-06-17,Ravinder Kumar,MONTHLY,30,L4,L4,NV,AFC,GATE,GATE29,29.0,NV-GATE-29,1.0,1.0,SAMSUNG,Concourse,NaT,NaT,NaN,baseline
